In [1]:
import ssl
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context

try:
    import httpx
    class PatchedClient(httpx.Client):
        def __init__(self, *args, **kwargs):
            kwargs['verify'] = False
            super().__init__(*args, **kwargs)
    class PatchedAsyncClient(httpx.AsyncClient):
        def __init__(self, *args, **kwargs):
            kwargs['verify'] = False
            super().__init__(*args, **kwargs)
    httpx.Client = PatchedClient
    httpx.AsyncClient = PatchedAsyncClient
except Exception:
    pass

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_community.document_loaders import PyPDFLoader
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [3]:

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))

model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

EMBEDDING_DEPLOYMENT_NAME = os.getenv('AZURE_EMBEDDING_DEPLOYMENT_NAME', 'BT-Embedding')
EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_APIKEY', AZURE_OPENAI_API_KEY)
EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT', AZURE_OPENAI_ENDPOINT)

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=EMBEDDING_ENDPOINT,
    api_key=EMBEDDING_API_KEY,
    azure_deployment=EMBEDDING_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
)

In [4]:
pdf_files = [
    "paper-1.pdf",
    "paper-2.pdf",
    "paper-3.pdf"
]

docs_raw = []
for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        docs_raw.extend(loader.load())
        print(f"✅ Loaded: {pdf}")
    except Exception as e:
        print(f"❌ Failed: {pdf} → {e}")

print(f"Total pages loaded: {len(docs_raw)}")

✅ Loaded: paper-1.pdf
✅ Loaded: paper-2.pdf
✅ Loaded: paper-3.pdf
Total pages loaded: 181


In [ ]:
from openai import OpenAI
import chromadb
from chromadb.config import Settings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os

load_dotenv()

# -------------------
# 1. Config
# -------------------
EMBEDDING_API_KEY = os.getenv('EMBEDDING_API_KEY', '')
EMBEDDING_DEPLOYMENT_NAME = os.getenv('EMBEDDING_DEPLOYMENT_NAME', 'BT-Embedding')
EMBEDDING_ENDPOINT = os.getenv('EMBEDDING_ENDPOINT', '')
DB_PATH = "./chroma_db"
COLLECTION_NAME = "pdf_chunks"

# -------------------
# 2. Clients
# -------------------
azure_client = OpenAI(
    base_url=EMBEDDING_ENDPOINT,
    api_key=EMBEDDING_API_KEY
)

chroma_client = chromadb.Client(Settings(persist_directory=DB_PATH))
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

# -------------------
# 3. Embedding Function
# -------------------
def get_embedding(text: str) -> list:
    response = azure_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT_NAME
    )
    return response.data[0].embedding

# -------------------
# 4. Load PDFs
# -------------------
pdf_files = [
    "paper-1.pdf",
    "paper-2.pdf",
    "paper-3.pdf"
]
docs_raw = []
for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        docs_raw.extend(loader.load())
        print(f"✅ Loaded: {pdf}")
    except Exception as e:
        print(f"❌ Failed: {pdf} → {e}")

print(f"Total pages loaded: {len(docs_raw)}")

# -------------------
# 5. Split
# -------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = splitter.split_documents(docs_raw)
print(f"Total chunks after splitting: {len(splits)}")

# -------------------
# 6. Embed + Store in Chroma
# -------------------
batch_size = 50  # avoid rate limits

for i in range(0, len(splits), batch_size):
    batch = splits[i: i + batch_size]

    ids        = [f"chunk_{i + j}" for j in range(len(batch))]
    texts      = [doc.page_content for doc in batch]
    metadatas  = [doc.metadata for doc in batch]
    embeddings = [get_embedding(text) for text in texts]

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )
    print(f" Stored batch {i // batch_size + 1} — chunks {i} to {min(i + batch_size, len(splits))}")

print(f"    All {len(splits)} chunks stored in Chroma at '{DB_PATH}'")

# -------------------
# 7. Test Retrieval
# -------------------
query = "What is attention mechanism?"
query_embedding = get_embedding(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("\n--- Test Retrieval Results ---")
for i, doc in enumerate(results['documents'][0]):
    source = results['metadatas'][0][i].get('source', 'unknown')
    print(f"\nChunk {i+1} (source: {source}):")
    print(doc[:300])

✅ Loaded: paper-1.pdf
✅ Loaded: paper-2.pdf
✅ Loaded: paper-3.pdf
Total pages loaded: 181
Total chunks after splitting: 631
✅ Stored batch 1 — chunks 0 to 50
✅ Stored batch 2 — chunks 50 to 100
✅ Stored batch 3 — chunks 100 to 150
✅ Stored batch 4 — chunks 150 to 200
✅ Stored batch 5 — chunks 200 to 250
✅ Stored batch 6 — chunks 250 to 300
✅ Stored batch 7 — chunks 300 to 350
✅ Stored batch 8 — chunks 350 to 400
✅ Stored batch 9 — chunks 400 to 450
✅ Stored batch 10 — chunks 450 to 500
✅ Stored batch 11 — chunks 500 to 550
✅ Stored batch 12 — chunks 550 to 600
✅ Stored batch 13 — chunks 600 to 631
✅ All 631 chunks stored in Chroma at './chroma_db'

--- Test Retrieval Results ---

Chunk 1 (source: paper-1.pdf):
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effect

In [15]:
query = "What is Reinforcement Learning & Alignment?"
query_embedding = get_embedding(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("\n--- Test Retrieval Results ---")
for i, doc in enumerate(results['documents'][0]):
    source = results['metadatas'][0][i].get('source', 'unknown')
    print(f"\nChunk {i+1} (source: {source}):")
    print(doc)


--- Test Retrieval Results ---

Chunk 1 (source: paper-3.pdf):
URL https://openai.com/blog/our-approach-to-alignment-research .
[70] Joseph Carlsmith. Is power-seeking AI an existential risk? ArXiv, abs/2206.13353, 2022.
[71] Amelia Glaese, Nat McAleese, Maja Tr˛ ebacz, John Aslanides, Vlad Firoiu, Timo Ewalds, Mari-
beth Rauh, Laura Weidinger, Martin Chadwick, Phoebe Thacker, Lucy Campbell-Gillingham,
Jonathan Uesato, Po-Sen Huang, Ramona Comanescu, Fan Yang, Abigail See, Sumanth
Dathathri, Rory Greig, Charlie Chen, Doug Fritz, Jaume Sanchez Elias, Richard Green, Soˇna
Mokrá, Nicholas Fernando, Boxi Wu, Rachel Foley, Susannah Young, Iason Gabriel, William
Isaac, John Mellor, Demis Hassabis, Koray Kavukcuoglu, Lisa Anne Hendricks, and Geoffrey
Irving. Improving alignment of dialogue agents via targeted human judgements. arXiv preprint
arXiv:2209.14375, 2022.
[72] Ethan Perez, Saffron Huang, H. Francis Song, Trevor Cai, Roman Ring, John Aslanides, Amelia
Glaese, Nat McAleese, and Geoff

In [ ]:
#Retriever to retriever tools

from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    
)


In [ ]:
# -------------------
# 6. RAG Tool
# -------------------
@tool
def retrieve_from_pdfs(query: str) -> str:
    """
    Search the research papers and return relevant content.
    Use this tool when the user asks anything about the PDFs,
    research papers, attention mechanism, reinforcement learning,
    deep learning, or any topic covered in the documents.
    """
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=4
    )

    if not results['documents'][0]:
        return "No relevant content found in the documents."

    output = []
    for i, doc in enumerate(results['documents'][0]):
        source = results['metadatas'][0][i].get('source', 'unknown')
        output.append(f"[Source: {source}]\n{doc}")

    return "\n\n---\n\n".join(output)

# -------------------
# 7. Tools + Model
# -------------------
tools = [retrieve_from_pdfs]
model_with_tools = model.bind_tools(tools)

# -------------------
# 8. State
# -------------------
class RAGState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# -------------------
# 9. Nodes
# -------------------
SYSTEM_PROMPT = """You are a helpful research assistant with access to research papers.
When the user asks a question, ALWAYS use the retrieve_from_pdfs tool first to search 
the documents before answering. Base your answers on the retrieved content.
If the retrieved content doesn't contain the answer, say so honestly."""

def chat_node(state: RAGState):
    messages = state['messages']

    # inject system prompt if first message
    from langchain_core.messages import SystemMessage
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

# -------------------
# 10. Graph
# -------------------
graph = StateGraph(RAGState)
graph.add_node("chat", chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat")
graph.add_conditional_edges("chat", tools_condition)
graph.add_edge("tools", "chat")

checkpointer = InMemorySaver()
chatbot = graph.compile(checkpointer=checkpointer)

# -------------------
# 11. Chat Loop
# -------------------
print("\n🤖 Agentic RAG ready! Type 'exit' to quit.\n")
config = {"configurable": {"thread_id": "rag-session-1"}}

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    if not user_input:
        continue

    response = chatbot.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    print(f"\n🤖 Assistant: {response['messages'][-1].content}\n")